# mega-log-analysis — interactive

Exploratory surface for the error-log campaign. The reproducible logic lives in
`mega_log_analysis.py`; this notebook **imports** it rather than re-implementing it, so a
classifier change cannot silently disagree between the two.

## Running it

The flake's `pythonEnv` provides only `pytest`, `pyyaml` and `rich` — no jupyter, pandas or
matplotlib. `jupyter` is on PATH, the rest come from `uv`:

```bash
uv run --with jupyter --with pandas --with matplotlib \
  jupyter notebook scripts/mega-log-analysis/analysis.ipynb
```

Every cell degrades to plain text if pandas/matplotlib are absent, so the notebook is still
readable under a bare `jupyter`.

**Read-only.** Nothing here mutates the cluster or files tickets.

In [ ]:
import json, sys
from pathlib import Path

TOOL = Path.cwd() if (Path.cwd() / "mega_log_analysis.py").exists() else Path("scripts/mega-log-analysis")
sys.path.insert(0, str(TOOL.resolve()))
import mega_log_analysis as mla

try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False
    print("pandas absent — falling back to plain text. See the header cell for the uv command.")

snaps = sorted((TOOL / "baseline").glob("census-*.json"))
assert snaps, "baseline/ is empty — run: task dev:logs:census -- --save"
data = json.loads(snaps[-1].read_text())
print(f"{snaps[-1].name}  window={data['window_hours']}h  "
      f"total={data['total_errors']:,}  services={len(data['services'])}  "
      f"signatures={len(data['signatures'])}")

## 1. Where are the errors?

Per service, from the committed snapshot. This is the raw `SeverityText` view — **before**
triage, so it still includes the ~40% that are not really errors.

In [ ]:
svcs = sorted(data["services"], key=lambda s: -s["errors"])
if HAVE_PANDAS:
    df = pd.DataFrame(svcs)
    df["share_%"] = (df.errors / df.errors.sum() * 100).round(1)
    display(df.head(20))
else:
    for s in svcs[:20]:
        print(f"{s['errors']:>7,}  {s['per_sec']:>7.3f}/s  {s['service']:<32} {s['namespace']}")

## 2. How much of this is real?

`triage` classifies each signature. `misclassified` means the body states its own level and that
level is not an error — those are ingest-pipeline bugs, not app bugs, and filing them as service
tickets sends an agent to investigate `"Saved checkpoint for VPA"`.

In [ ]:
from collections import Counter

verdicts, reasons = Counter(), Counter()
for s in data["signatures"]:
    verdicts[s["verdict"]] += s["count"]
    reasons[s["reason"]] += s["count"]
total = sum(verdicts.values())

for v, n in verdicts.most_common():
    print(f"  {v:<16} {n:>8,}  {n/total*100:>5.1f}%")
print("\n  why:")
for r, n in reasons.most_common(8):
    print(f"    {n:>8,}  {r}")

## 3. Concentration — how few fixes buy how much?

The campaign is only tractable because the distribution is steep. If the top-N curve were flat,
the right answer would be sampling, not a ticket tree.

In [ ]:
real = sorted([s for s in data["signatures"] if s["verdict"] == "real"],
              key=lambda s: -s["count"])
rtot = sum(s["count"] for s in real) or 1
cum, curve = 0, []
for i, s in enumerate(real, 1):
    cum += s["count"]
    curve.append((i, cum / rtot * 100))

for n in (5, 10, 20, 30, 50):
    if n <= len(curve):
        print(f"  top {n:>3} real signatures -> {curve[n-1][1]:5.1f}% of {rtot:,} real errors")

try:
    import matplotlib.pyplot as plt
    xs, ys = zip(*curve)
    plt.figure(figsize=(7, 3.5))
    plt.plot(xs, ys)
    plt.axhline(90, ls="--", lw=0.8)
    plt.xlabel("signatures fixed (most frequent first)")
    plt.ylabel("% of real errors removed")
    plt.title("Cumulative coverage")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("\n  (matplotlib absent — numbers above are the whole story)")

## 4. The actionable tree

Exactly what `task dev:logs:tickets` would file: services above the floor, and their real
signatures. Change `MIN` to see how the floor moves the ticket count.

In [ ]:
MIN = 60  # errors per window for a signature to earn a ticket

tier = mla._service_tier(data, MIN)
leaves = 0
for svc in tier:
    sigs = [s for s in svc["sigs"] if s["count"] >= MIN]
    leaves += len(sigs)
    print(f"L2 {svc['service']:<30} {svc['errors']:>6,} err  ({svc['namespace']})")
    for s in sigs:
        print(f"     L3 {s['count']:>6,}  {s['sig_id']}  {s['sig'][:78]}")
print(f"\n{len(tier)} sub-epics · {leaves} leaf tickets at floor {MIN}")

## 5. Ad-hoc: drill into one signature

Live query against ClickHouse. Paste a `sig_id` from the tree above.

In [ ]:
SIG_ID = real[0]["sig_id"] if real else None

s = next((x for x in data["signatures"] if x["sig_id"] == SIG_ID), None)
if not s:
    print("set SIG_ID to one from the tree above")
else:
    print(f"{s['service']}  {s['count']:,}/window\n{s['sig']}\n")
    quoted = s["sig"].replace(chr(39), chr(39) * 2)
    rows = mla.tsv(
        "SELECT substring(Body, 1, 300) FROM default.otel_logs"
        f" WHERE TimestampTime > now() - INTERVAL {data['window_hours']} HOUR"
        f" AND SeverityText IN ({mla._sev_list()})"
        f" AND ServiceName = '{s['service']}'"
        f" AND {mla.SIG_EXPR} = '{quoted}' LIMIT 3")
    for r in rows:
        print("  " + r[0][:300])

## 6. Burn-down across committed baselines

The campaign metric. Needs ≥2 snapshots in `baseline/`; each `task dev:logs:census -- --save`
adds one.

In [ ]:
hist = []
for p in snaps:
    d = json.loads(p.read_text())
    real_n = sum(x["count"] for x in d["signatures"] if x["verdict"] == "real")
    hist.append((d["captured_utc"], d["total_errors"], real_n, d["window_hours"]))

print(f"  {'captured (UTC)':<28} {'total':>8} {'real':>8} {'real/sec':>9}")
for ts, tot, rl, w in hist:
    print(f"  {ts[:19]:<28} {tot:>8,} {rl:>8,} {rl/(w*3600):>9.2f}")
if len(hist) < 2:
    print("\n  only one snapshot — no trend yet")
else:
    d = hist[-1][2] - hist[0][2]
    print(f"\n  real errors {hist[0][2]:,} -> {hist[-1][2]:,}  ({d:+,})")